## Status (created 2026-08-04) -- READ BEFORE RUNNING

**Pancreas only, dimension 20.** Third point on the dimension axis, after d=8
(`batch_correct_then_cluster_baselines.ipynb`) and d=50 (`harmony_default_dim.ipynb`).
Same shared helper module, same downstream pipeline -- only the dimension differs.

Separate notebook so it can run in its own Colab session without touching the other two.
Results land in their own folders (`X_harmony_d20`, `..._LD20_...`); the d=8 and d=50 runs
are read-only here and appear in the comparison tables alongside the new rows.

Everything is load-if-exists: re-running any cell after a disconnect reloads rather than
recomputes.

# Harmony at 20 PCs + scProto at a 20-dim latent -- pancreas

**Why 20.** It is the harmony authors' own default: the R package's `HarmonyMatrix`
entry point is `do_pca=TRUE, npcs=20`, and their quickstart vignette uses the top 20 PCs.
(`harmonypy` itself performs no PCA -- it takes whatever embedding you hand it -- and the
50 used in the sibling notebook is scanpy's / Seurat's convention, which is the number
Reviewer nG29 named.) So 20 and 50 are both defensible "defaults", from different sources.

**What this notebook adds.** With d=8, d=20 and d=50 on the same dataset we can see the
shape of the curve rather than two isolated points:

- **For Harmony**, whether its rare-cell jump between 8 and 50 (F1 0.32 -> 0.66 on
  pancreas) is gradual or a threshold effect.
- **For scProto**, whether the batch-entropy drop seen at d=50 (weighted 1.021 at d=8 ->
  0.865 at d=50, median per-metacell entropy hitting 0.000) grows with latent dimension.
  That is the mechanism claim in `experiment-results/harmony_default_dim_results.md`
  (Rule 4) -- d=20 is what tests it.

Both are reported with the paper's own rare-cell metrics (mean +- std across the 8
pancreas batches) and the paired one-sided Wilcoxon tests.

## Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q scarches faiss-gpu-cu12 scib-metrics
!pip install git+https://github.com/dpeerlab/SEACells.git --quiet --no-deps
!pip install numpy scipy --upgrade -q
!pip install -q palantir harmonypy
!pip install -q "numpy==1.26.4" "scipy==1.13.1"
!pip install --upgrade --force-reinstall numpy cupy-cuda12x
!pip install "numpy<2.3"

In [ ]:
# IMPORTANT: restart the runtime after this cell before running the cells below --
# numpy/scipy/anndata are C-extension linked, an in-process upgrade alone won't
# reliably take effect on already-imported modules.

In [ ]:
# Ground truth for "did the install cell above actually work" -- pip's own log is noisy
# (resolver backtracking prints errors for discarded candidate versions even on a fully
# successful install), so importing every package is the real test.
_checks = {
    'numpy': 'numpy', 'scipy': 'scipy', 'anndata': 'anndata', 'scanpy': 'scanpy',
    'scarches': 'scarches', 'scvi-tools': 'scvi', 'seacells': 'SEACells',
    'palantir': 'palantir', 'scib-metrics': 'scib_metrics', 'leidenalg': 'leidenalg',
    'python-igraph': 'igraph', 'umap-learn': 'umap', 'harmonypy': 'harmonypy',
    'faiss-cpu': 'faiss',
}
_failed = []
for pkg_name, import_name in _checks.items():
    try:
        mod = __import__(import_name)
        print(f"  OK   {pkg_name:16s} (import {import_name}, version {getattr(mod, '__version__', '?')})")
    except Exception as e:
        _failed.append(pkg_name)
        print(f"  FAIL {pkg_name:16s} (import {import_name}): {type(e).__name__}: {e}")

if _failed:
    print(f"\n{len(_failed)} package(s) failed to import: {_failed} -- re-run that "
          f"package's specific pip install line above.")
else:
    print(f"\nAll {len(_checks)} packages import cleanly -- safe to continue.")

In [ ]:
%run /content/drive/MyDrive/codes/interpretable-prototype/notebooks/nb_setup.py

In [ ]:
import os
from interpretable_ssl.datasets.dataset_configs import DATASETS
from interpretable_ssl.configs.paths import get_dataset_model_dir

print("extra imports ready")

## Config

Identical to the d=50 notebook except `DIM = 20`. `CVAE_EPOCHS=50` / `BATCH_SIZE=1024`
match the published Stage-1 hyperparameters; `TRAIN_EPOCHS=20` is the short Stage-2 budget
(the d=50 run early-stopped at epoch 9, so the cap was not the binding constraint there).

In [ ]:
DS = 'pancreas'
DATASETS_LIST = [DS]
ALL_DATASETS = ['pancreas', 'lung', 'pbmc-immune']   # used by the final section

DIM = 20                 # PCA components for Harmony AND scProto's latent_dims
CVAE_EPOCHS = 50
TRAIN_EPOCHS = 20
BATCH_SIZE = 1024
EVAL_FREQ = 3
PATIENCE = 6
UMAP_STEPS_PER_EPOCH = 500

SKIP_IF_EXISTS = True
CORRECTION_METHODS = ['harmony']
METHOD_DISPLAY_NAMES = {'harmony': f'Harmony d={DIM}'}
RUN_SEACELLS = True
RUN_LEIDEN = True

# Dimensions already on disk from the sibling notebooks -- pulled into the tables as
# read-only rows so all three dimensions appear side by side.
OTHER_DIMS = [8, 50]

dataset_display_names = {'pancreas': 'Pancreas', 'lung': 'Lung', 'pbmc-immune': 'Immune'}
REF_DIM_NAME = f'scProto (d={DIM})'

## Helper functions

Same functions as the d=50 notebook, parameterized by `DIM`. Harmony goes through
`run_all_baselines_for_dataset(..., force_matched_n_comps=DIM)`; scProto goes through
`run_mc_task(..., trainer_kwargs={'latent_dims': DIM})`, the same entry point that
produced the paper's numbers.

In [ ]:
from interpretable_ssl.evaluation.batch_correct_baselines import run_all_baselines_for_dataset
from interpretable_ssl.experiments.tasks import run_mc_task, LAMBDA_PROTO_UMAP_PRECON
from interpretable_ssl.evaluation.metric_helpers.result_tables import extract_model_key


def run_harmony_at_dim(ds_id, dim=None):
    dim = DIM if dim is None else dim
    return run_all_baselines_for_dataset(
        ds_id, correction_methods=CORRECTION_METHODS, skip_if_exists=SKIP_IF_EXISTS,
        run_seacells=RUN_SEACELLS, run_leiden=RUN_LEIDEN, force_matched_n_comps=dim,
    )


def scproto_run_dirs(ds_id, latent_dim):
    """Run folders for scProto trained at `latent_dim`. latent_dims appears in the folder
    name as 'LD{n}' whenever it differs from the default 8, which is what keeps these runs
    separate from the published d=8 one on disk.
    """
    base = get_dataset_model_dir(ds_id)
    if not os.path.isdir(base):
        return []
    token = f'LD{latent_dim}'
    return sorted(
        d for d in os.listdir(base)
        if token in d and d.startswith('proto_umap') and os.path.isdir(os.path.join(base, d))
    )


def scproto_checkpoint_exists(ds_id, latent_dim):
    base = get_dataset_model_dir(ds_id)
    return any(
        os.path.exists(os.path.join(base, d, 'umap_checkpoint.pth'))
        for d in scproto_run_dirs(ds_id, latent_dim)
    )


def train_scproto_at_dim(ds_id, latent_dim=None):
    """scProto with latent_dims=latent_dim, everything else identical to the published
    configuration. Reloads an existing checkpoint instead of retraining.
    """
    latent_dim = DIM if latent_dim is None else latent_dim
    load_umap = scproto_checkpoint_exists(ds_id, latent_dim)
    print(f"=== [{ds_id}] scProto latent_dims={latent_dim} "
          f"[{'reloading existing checkpoint' if load_umap else 'training fresh'}] ===")
    t, res, mc = run_mc_task(
        ds_id,
        cvae_epochs=CVAE_EPOCHS, train_epochs=TRAIN_EPOCHS, eval_freq=EVAL_FREQ,
        patience=PATIENCE, batch_size=BATCH_SIZE,
        umap_steps_per_epoch=UMAP_STEPS_PER_EPOCH,
        lambda_config=LAMBDA_PROTO_UMAP_PRECON, affinity_type='arbf',
        load_umap=load_umap, trainer_kwargs={'latent_dims': latent_dim},
    )
    # Prove the latent dimension took effect rather than assuming the kwarg propagated:
    # a silent fallback to 8 would invalidate the comparison, so fail loudly.
    proto_shape = tuple(t.model.get_prototypes().shape)
    assert t.latent_dims == latent_dim, f"trainer.latent_dims={t.latent_dims}, expected {latent_dim}"
    assert proto_shape[1] == latent_dim, f"prototypes {proto_shape}, expected (K, {latent_dim})"
    assert f'LD{latent_dim}' in os.path.basename(t.get_dump_path()), (
        f"run folder carries no LD{latent_dim} token -- it would collide with another run"
    )
    print(f"[verified] latent_dims={t.latent_dims} | prototypes {proto_shape} | "
          f"run dir: {t.get_dump_path()}")
    return t, res, mc


def scproto_model_key(ds_ids, latent_dim):
    ds_ids = [ds_ids] if isinstance(ds_ids, str) else ds_ids
    keys = {
        extract_model_key(d, ds_id=ds)
        for ds in ds_ids for d in scproto_run_dirs(ds, latent_dim)
    }
    if not keys:
        print(f"No scProto LD{latent_dim} run on disk yet for {ds_ids}.")
        return None
    if len(keys) > 1:
        print(f"WARNING: multiple scProto LD{latent_dim} keys on disk: {sorted(keys)} -- "
              f"using '{sorted(keys)[0]}'.")
    return sorted(keys)[0]


print("helpers ready")

## Run: Harmony at d=20 (pancreas)

In [ ]:
harmony_d20 = run_harmony_at_dim(DS)

## Run: scProto at latent_dims=20 (pancreas)

In [ ]:
t20, res20, mc20 = train_scproto_at_dim(DS)
res20

## Results -- all three dimensions side by side

Rows: scProto at d=8 (published), d=20, d=50; Harmony at d=8, d=20, d=50 (each with
SEACells and Leiden on top); SEACells (PCA) as the paper's own baseline. Any row whose run
is not on disk is simply absent, not an error.

Rare-cell metrics are mean +- std across the 8 pancreas batches. Read F1 together with its
two halves (precision, recall) -- they move in opposite directions between the methods.

In [ ]:
from interpretable_ssl.evaluation.rebuttal_report import (
    build_model_keywords, dim_matched_read_only_keywords, render_full_comparison_report,
    RARE_CELL_SIG_METRICS,
)
from interpretable_ssl.evaluation.paper_figures import (
    rare_metric_significance_paired, graph_batch_significance_paired,
)

# Harmony at DIM (computed here) + the other dimensions read off disk. Distinct display
# names per dimension -- identical names would collapse into one row and silently drop a
# dimension from the comparison.
read_only = {}
for d in OTHER_DIMS:
    read_only.update(dim_matched_read_only_keywords('harmony', f'Harmony d={d}', matched_dim=d))

MODEL_KEYWORDS = build_model_keywords(
    CORRECTION_METHODS, METHOD_DISPLAY_NAMES, matched_dim=DIM, extra_read_only=read_only,
)

# scProto at every non-default latent dimension present on disk (the published d=8 run is
# already in MODEL_KEYWORDS as 'scProto').
for d in [DIM] + OTHER_DIMS:
    if d == 8:
        continue
    key = scproto_model_key(DS, d)
    if key:
        MODEL_KEYWORDS[key] = f'scProto (d={d})'

MODEL_KEYWORDS

In [ ]:
# scProto (published, d=8) as reference.
report = render_full_comparison_report(
    DATASETS_LIST, dataset_display_names, MODEL_KEYWORDS, ref_name='scProto',
)

In [ ]:
# Same-dimension comparison: scProto (d=20) vs Harmony d=20. Reuses the rare table
# computed above -- no recompute.
sig_rare = rare_metric_significance_paired(
    report['rare'], ref_name=REF_DIM_NAME, metrics=RARE_CELL_SIG_METRICS,
    dataset_display_names=dataset_display_names,
)
print(f"=== Rare-cell metrics, PAIRED one-sided Wilcoxon ({REF_DIM_NAME} > other) ===")
display(sig_rare)

sig_mod = graph_batch_significance_paired(
    DATASETS_LIST, MODEL_KEYWORDS, ref_name=REF_DIM_NAME,
    dataset_display_names=dataset_display_names,
)
print(f"\n=== Modularity per batch, PAIRED one-sided Wilcoxon ({REF_DIM_NAME} > other) ===")
display(sig_mod)

### Batch entropy vs. latent dimension -- the mechanism check

The d=50 result showed scProto's per-metacell batch entropy DROPPING as the latent grew
(weighted 1.021 at d=8 -> 0.865 at d=50, median 0.214 -> 0.000), i.e. more single-batch
metacells -- the proposed explanation for its lower cross-batch rare-cell grouping
(`experiment-results/harmony_default_dim_results.md`, Rule 4). d=20 sits between the two:
a monotone trend across 8 / 20 / 50 supports that explanation, a flat or non-monotone one
does not.

Median is the informative statistic here, not the mean -- a median of 0 means over half
the metacells contain cells from a single batch.

In [ ]:
import numpy as np
from interpretable_ssl.evaluation.paper_figures import _resolve_run_dir, _read_series


def batch_entropy_table(ds_ids, model_keywords):
    """Per-metacell batch entropy per method, straight from each run's saved
    batch_entropy_per_mc.csv -- no recompute. frac_single_batch is the share of metacells
    containing cells from exactly one batch, which is what a median of 0 is really saying.
    """
    ds_ids = [ds_ids] if isinstance(ds_ids, str) else ds_ids
    rows = []
    for ds_id in ds_ids:
        for keyword, name in model_keywords.items():
            run_dir = _resolve_run_dir(ds_id, keyword, prefer_csv='batch_entropy_per_mc.csv')
            if run_dir is None:
                continue
            ser = _read_series(os.path.join(run_dir, 'batch_entropy_per_mc.csv'))
            if ser is None:
                continue
            v = ser.values.astype(float)
            rows.append({
                'dataset': dataset_display_names.get(ds_id, ds_id), 'method': name,
                'n_metacells': len(v),
                'entropy_mean': round(float(np.mean(v)), 3),
                'entropy_median': round(float(np.median(v)), 3),
                'frac_single_batch': round(float((v <= 1e-9).mean()), 3),
            })
    return pd.DataFrame(rows).sort_values(['dataset', 'entropy_median'], ascending=[True, False])


display(batch_entropy_table(DATASETS_LIST, MODEL_KEYWORDS))

### Realized cluster/metacell count vs. K

Confirms Harmony's downstream SEACells/Leiden landed at scProto's own `num_prototypes`
(220 for pancreas), so every method is compared at equal K. Reads saved outputs only.

In [ ]:
from interpretable_ssl.evaluation.batch_correct_baselines import get_realized_seacell_count

target_k = DATASETS[DS]['num_prototypes']
harmony_tag = f'X_harmony_d{DIM}'

df_k = load_task1_multi(DATASETS_LIST, metrics=['n_clusters', 'resolution'])
if not df_k.empty:
    is_leiden = df_k.index.get_level_values('run').str.startswith(f'leiden_{harmony_tag}')
    if is_leiden.any():
        out = df_k[is_leiden].copy()
        out['target_k'] = target_k
        out['matches_target'] = out['n_clusters'] == target_k
        display(out)
    else:
        print(f"No leiden_{harmony_tag} run found yet.")

n_actual = get_realized_seacell_count(DS, harmony_tag)
print(f"SEACells on {harmony_tag}: realized={n_actual}, target={target_k}, "
      f"matches={n_actual is not None and abs(n_actual - target_k) <= 0.05 * target_k}")

### Embedding-only rare-cell affinity purity (no clustering)

One ARBF affinity graph built directly on each embedding; for each locally-rare-type cell,
the fraction of its total affinity mass going to same-type cells. Ran automatically inside
the Harmony cell above. The `dim` column separates the d=8 / d=20 / d=50 Harmony rows.

In [ ]:
from interpretable_ssl.evaluation.batch_correct_baselines import load_and_compare_affinity_purity

load_and_compare_affinity_purity(DATASETS_LIST, dataset_display_names=dataset_display_names)

In [ ]:
# Flat CSV of the rare-cell table -- convenient for pasting numbers into the write-up.
save_path = '/content/drive/MyDrive/models/rare_metrics_harmony_d20_pancreas.csv'
report['rare'].to_csv(save_path)
print(f"saved to {save_path}")
report['rare']

## Lung and Immune -- same two runs

Harmony at d=20 and scProto at latent_dims=20 for the remaining two datasets. Each cell is
independent and load-if-exists, so they can be run in any order or resumed after a
disconnect. Lung carries the most weight of the three (15 batches vs. pancreas' 8 and
immune's 5), so its rare-cell numbers are the ones most likely to reach significance.

### Lung -- Harmony at d=20

In [ ]:
lung_harmony_d20 = run_harmony_at_dim('lung')

### Lung -- scProto at latent_dims=20

In [ ]:
t_lung20, res_lung20, mc_lung20 = train_scproto_at_dim('lung')
res_lung20

### Immune (PBMC) -- Harmony at d=20

In [ ]:
immune_harmony_d20 = run_harmony_at_dim('pbmc-immune')

### Immune (PBMC) -- scProto at latent_dims=20

In [ ]:
t_imm20, res_imm20, mc_imm20 = train_scproto_at_dim('pbmc-immune')
res_imm20

## All three datasets -- final tables

Everything above, re-rendered across pancreas, lung and immune together. This is the
section to read the reported numbers off.

The **rare-cell table** is the key one: coverage, recall, precision, homogeneity,
cross-batch homogeneity and macro F1, each as mean +- std across that dataset's batches,
followed by the paired one-sided Wilcoxon tests against both references (scProto d=8, the
published model, and scProto d=20, the dimension-matched one).

In [ ]:
# Same keyword map as above, resolved over all three datasets.
read_only_all = {}
for d in OTHER_DIMS:
    read_only_all.update(dim_matched_read_only_keywords('harmony', f'Harmony d={d}', matched_dim=d))

MODEL_KEYWORDS_ALL = build_model_keywords(
    CORRECTION_METHODS, METHOD_DISPLAY_NAMES, matched_dim=DIM, extra_read_only=read_only_all,
)
for d in [DIM] + OTHER_DIMS:
    if d == 8:
        continue
    key = scproto_model_key(ALL_DATASETS, d)
    if key:
        MODEL_KEYWORDS_ALL[key] = f'scProto (d={d})'

MODEL_KEYWORDS_ALL

In [ ]:
# Full report across all three datasets, scProto (published, d=8) as reference.
# Includes Table 1, Table 2 and the rare-cell table + significance tests.
report_all = render_full_comparison_report(
    ALL_DATASETS, dataset_display_names, MODEL_KEYWORDS_ALL, ref_name='scProto',
)

In [ ]:
# Same-dimension reference (scProto d=20), all three datasets.
sig_rare_all = rare_metric_significance_paired(
    report_all['rare'], ref_name=REF_DIM_NAME, metrics=RARE_CELL_SIG_METRICS,
    dataset_display_names=dataset_display_names,
)
print(f"=== Rare-cell metrics, PAIRED one-sided Wilcoxon ({REF_DIM_NAME} > other) ===")
display(sig_rare_all)

sig_mod_all = graph_batch_significance_paired(
    ALL_DATASETS, MODEL_KEYWORDS_ALL, ref_name=REF_DIM_NAME,
    dataset_display_names=dataset_display_names,
)
print(f"\n=== Modularity per batch, PAIRED one-sided Wilcoxon ({REF_DIM_NAME} > other) ===")
display(sig_mod_all)

In [ ]:
# Rare-cell metrics only, as a compact per-dataset view (mean +- std merged by show_table).
from interpretable_ssl.evaluation.rebuttal_report import RARE_CELL_METRICS

show_table(report_all['rare'], metrics=RARE_CELL_METRICS,
           dataset_display_names=dataset_display_names)

In [ ]:
# Batch entropy vs. latent dimension, all three datasets -- the mechanism check from the
# pancreas section, repeated where there is more data behind it.
display(batch_entropy_table(ALL_DATASETS, MODEL_KEYWORDS_ALL))

In [ ]:
load_and_compare_affinity_purity(ALL_DATASETS, dataset_display_names=dataset_display_names)

In [ ]:
# Flat CSV of the rare-cell table for all three datasets.
save_path_all = '/content/drive/MyDrive/models/rare_metrics_harmony_d20_all.csv'
report_all['rare'].to_csv(save_path_all)
print(f"saved to {save_path_all}")
report_all['rare']